# Proyecto final · Olist: del pedido a la reseña

**Equipo:** Ruiz Melissa, Rodrigo y Cesar

Esta plantilla solo fija **el orden y los encabezados** (no los borres: se usan para calificar). El código y las decisiones son de ustedes. Pueden agregar todas las celdas que necesiten dentro de cada parte.

Antes de entregar: *Entorno de ejecución › Reiniciar y ejecutar todo*. El notebook debe correr de principio a fin sin errores.

> **Cómo leer este notebook.** Todas las cifras de las respuestas se imprimen desde las salidas de las celdas (celdas que generan el texto con los números calculados), así cada número del texto sale de una salida del propio notebook.

## 0. Conexión

Usen el usuario de solo lectura del curso. Al correr la celda, escriban **solo la contraseña** que les dio el docente (no la cadena completa). Los datos están en el esquema `final`.

In [ ]:
import warnings
from getpass import getpass
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sqlalchemy.pool import NullPool

warnings.filterwarnings("ignore")
HOST = "aws-0-us-west-2.pooler.supabase.com"
USUARIO = "alumnos_lectura.ashlmfqpqutuoqtcmzqp"
CLAVE = getpass("Contraseña: ")          # la contraseña NO se guarda en el notebook
motor = create_engine(f"postgresql+psycopg2://{USUARIO}:{quote_plus(CLAVE)}@{HOST}:5432/postgres", poolclass=NullPool)
pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'final' ORDER BY 1", motor)

## Parte 1 · Datos

Escriban la consulta SQL que produce **una fila por pedido entregado y con reseña** con las 18 columnas del contrato (ver enunciado, sección "Contrato"). Expliquen en Markdown cada decisión: qué hicieron con pedidos de varios artículos, varios pagos y varias reseñas.

In [ ]:
CONSULTA = """
WITH articulos AS (          -- 1 fila por pedido: agregados de todos sus artículos
    SELECT oi.order_id,
           COUNT(*)                            AS n_items,
           COUNT(DISTINCT oi.seller_id)        AS n_vendedores,
           SUM(oi.price)::float                AS precio_total,
           SUM(oi.freight_value)::float        AS flete_total,
           SUM(p.product_weight_g)::float      AS peso_total_g,
           AVG(p.product_photos_qty)::float    AS fotos_promedio
    FROM final.order_items oi
    LEFT JOIN final.products p ON p.product_id = oi.product_id
    GROUP BY oi.order_id
),
articulo1 AS (               -- categoría y estado del vendedor del artículo 1
    SELECT oi.order_id,
           p.product_category_name AS categoria,
           s.seller_state          AS estado_vendedor
    FROM final.order_items oi
    LEFT JOIN final.products p ON p.product_id = oi.product_id
    LEFT JOIN final.sellers  s ON s.seller_id  = oi.seller_id
    WHERE oi.order_item_id = 1
),
pago_principal AS (          -- tipo de pago del pago de mayor monto
    SELECT DISTINCT ON (order_id) order_id, payment_type AS tipo_pago
    FROM final.order_payments
    ORDER BY order_id, payment_value DESC, payment_sequential
),
cuotas AS (                  -- máximo de cuotas del pedido
    SELECT order_id, MAX(payment_installments) AS cuotas_max
    FROM final.order_payments
    GROUP BY order_id
),
resena AS (                  -- una sola reseña: la de review_answer_timestamp más reciente
    SELECT DISTINCT ON (order_id) order_id,
           CASE WHEN review_score::int <= 2 THEN 1 ELSE 0 END AS mala_resena
    FROM final.order_reviews
    ORDER BY order_id, review_answer_timestamp DESC NULLS LAST
)
SELECT o.order_id,
       o.order_purchase_timestamp,
       o.order_approved_at,
       o.order_delivered_carrier_date,
       o.order_delivered_customer_date,
       o.order_estimated_delivery_date,
       c.customer_state AS estado_cliente,
       a.n_items, a.n_vendedores, a.precio_total, a.flete_total,
       a.peso_total_g, a.fotos_promedio,
       a1.categoria, a1.estado_vendedor,
       pp.tipo_pago, cu.cuotas_max,
       r.mala_resena
FROM final.orders o
JOIN final.customers c       ON c.customer_id = o.customer_id
JOIN articulos a             ON a.order_id    = o.order_id
LEFT JOIN articulo1 a1       ON a1.order_id   = o.order_id
LEFT JOIN pago_principal pp  ON pp.order_id   = o.order_id
LEFT JOIN cuotas cu          ON cu.order_id   = o.order_id
JOIN resena r                ON r.order_id    = o.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
ORDER BY o.order_id
"""

pedidos = pd.read_sql(CONSULTA, motor)

FECHAS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in FECHAS:
    pedidos[col] = pd.to_datetime(pedidos[col], utc=True).dt.tz_localize(None)

pedidos.head()

Esta celda verifica el contrato. Debe imprimir `True`:

In [ ]:
CONTRATO = ["order_id", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
            "order_delivered_customer_date", "order_estimated_delivery_date", "estado_cliente", "n_items",
            "n_vendedores", "precio_total", "flete_total", "peso_total_g", "fotos_promedio", "categoria",
            "estado_vendedor", "tipo_pago", "cuotas_max", "mala_resena"]
print("Contrato completo:", list(pedidos.columns) == CONTRATO, "|", len(pedidos), "pedidos")

### Qué decidimos con pedidos de varios artículos, varios pagos y varias reseñas, y por qué

La celda siguiente mide cuántos pedidos tienen cada situación, para dimensionar el impacto de cada decisión.

In [ ]:
multi = pd.read_sql("""
SELECT
  (SELECT COUNT(*) FROM (SELECT order_id FROM final.order_items    GROUP BY 1 HAVING COUNT(*) > 1) t)                    AS pedidos_varios_articulos,
  (SELECT COUNT(*) FROM (SELECT order_id FROM final.order_items    GROUP BY 1 HAVING COUNT(DISTINCT seller_id) > 1) t)   AS pedidos_varios_vendedores,
  (SELECT COUNT(*) FROM (SELECT order_id FROM final.order_payments GROUP BY 1 HAVING COUNT(*) > 1) t)                    AS pedidos_varios_pagos,
  (SELECT COUNT(*) FROM (SELECT order_id FROM final.order_payments GROUP BY 1 HAVING COUNT(DISTINCT payment_type) > 1) t) AS pedidos_varios_tipos_pago,
  (SELECT COUNT(*) FROM (SELECT order_id FROM final.order_reviews  GROUP BY 1 HAVING COUNT(*) > 1) t)                    AS pedidos_varias_resenas
""", motor).T.rename(columns={0: "pedidos"})
multi

**1. Pedidos con varios artículos (`order_items`).**
- *Cómo:* la subconsulta `articulos` agrupa por `order_id` y calcula `COUNT(*)` (n_items), `COUNT(DISTINCT seller_id)` (n_vendedores), `SUM` de precio, flete y peso, y `AVG` de fotos. La subconsulta `articulo1` toma la categoría y el estado del vendedor **solo del artículo 1** (`order_item_id = 1`), como define el contrato.
- *Por qué:* el modelo predice por pedido, así que necesitamos exactamente una fila por pedido. Sumar conserva el tamaño real del envío (un paquete más grande o con varios vendedores tiene más puntos de falla: envíos parciales, retrasos de un vendedor). Tomar el artículo 1 para las variables categóricas evita duplicar filas; es una aproximación razonable porque la gran mayoría de pedidos tiene un solo artículo (ver tabla de arriba).

**2. Pedidos con varios pagos (`order_payments`).**
- *Cómo:* `pago_principal` usa `DISTINCT ON (order_id)` ordenado por `payment_value DESC` (y `payment_sequential` para desempatar), y se queda con el tipo de pago de **mayor monto**. `cuotas` toma el `MAX(payment_installments)`.
- *Por qué:* el contrato pide el tipo de pago de mayor monto: si un cliente pagó casi todo con tarjeta y un saldo pequeño con voucher, lo que describe su compra es la tarjeta. El máximo de cuotas refleja el mayor compromiso financiero que asumió el cliente en ese pedido.

**3. Pedidos con varias reseñas (`order_reviews`).**
- *Cómo:* `resena` usa `DISTINCT ON (order_id)` ordenado por `review_answer_timestamp DESC NULLS LAST` y binariza: `mala_resena = 1` si `review_score` es 1 o 2.
- *Por qué:* la reseña más reciente es la opinión final del cliente (si la corrigió, vale la última). Usar una sola evita duplicar pedidos y que un mismo pedido cuente dos veces en el objetivo. La reseña **solo** se usa para construir la etiqueta, nunca como variable de entrada.

**4. Filtros.** `order_status = 'delivered'` y `order_delivered_customer_date IS NOT NULL`, y `JOIN` (no `LEFT JOIN`) con `resena`: solo pedidos entregados, con fecha de entrega conocida y con reseña, exactamente lo que define el contrato. `ORDER BY order_id` hace que el resultado sea reproducible.

## Limpieza y diccionario de variables

Faltantes, valores raros y qué hicieron con ellos. Tabla con cada variable que crean en `preparacion.py`: nombre, cómo se calcula y qué significa.

In [ ]:
print("Filas y columnas:", pedidos.shape)
print("¿order_id único?:", pedidos["order_id"].is_unique)
print("Tasa de mala reseña:", round(pedidos["mala_resena"].mean(), 3))
print("Periodo de compra:", pedidos["order_purchase_timestamp"].min().date(), "a", pedidos["order_purchase_timestamp"].max().date())

print("\n--- Faltantes (solo columnas con faltantes) ---")
faltantes = pedidos.isna().sum()
faltantes = faltantes[faltantes > 0]
print(pd.DataFrame({"faltantes": faltantes, "%": (faltantes / len(pedidos) * 100).round(2)}))

print("\n--- Resumen numérico ---")
print(pedidos[["n_items", "n_vendedores", "precio_total", "flete_total", "peso_total_g",
               "fotos_promedio", "cuotas_max"]].describe().T.round(2))

print("\n--- Valores raros ---")
dias_entrega = (pedidos["order_delivered_customer_date"] - pedidos["order_purchase_timestamp"]).dt.days
raros = {
    "Entrega antes de la compra": (dias_entrega < 0).sum(),
    "Entrega a más de 100 días": (dias_entrega > 100).sum(),
    "precio_total <= 0": (pedidos["precio_total"] <= 0).sum(),
    "flete_total == 0 (envío gratis)": (pedidos["flete_total"] == 0).sum(),
    "peso_total_g == 0": (pedidos["peso_total_g"] == 0).sum(),
    "cuotas_max == 0": (pedidos["cuotas_max"] == 0).sum(),
    "Entrega al cliente antes que al transportista": (pedidos["order_delivered_customer_date"] < pedidos["order_delivered_carrier_date"]).sum(),
    "Aprobación antes de la compra": (pedidos["order_approved_at"] < pedidos["order_purchase_timestamp"]).sum(),
}
for k, v in raros.items():
    print(f"{k}: {v}")

### Qué hicimos con faltantes y valores raros

**Faltantes**
| Columna | Causa probable | Tratamiento |
|---|---|---|
| `order_approved_at`, `order_delivered_carrier_date` | Registro operativo incompleto (pocos casos) | Se conservan. Las diferencias de días que dependen de ellas quedan `NaN` y el pipeline las imputa con la **mediana** (`SimpleImputer(strategy="median")`) aprendida solo con los datos de entrenamiento. |
| `peso_total_g`, `fotos_promedio`, `categoria` | Productos sin ficha completa en `products` (el `LEFT JOIN` conserva el pedido) | Numéricas → mediana dentro del pipeline. `categoria` → etiqueta `"desconocido"` en `preparacion.py` (puede ser informativa: productos mal catalogados). |
| `tipo_pago`, `cuotas_max` | Pedido sin registro en `order_payments` | `tipo_pago` → `"desconocido"`; `cuotas_max` → mediana. |

No eliminamos filas: el modelo debe predecir para cualquier pedido del contrato (incluidos los ocultos), así que el tratamiento de faltantes vive **dentro del pipeline**, no en una limpieza previa que la API no podría repetir.

**Valores raros**
- *Entregas a más de 100 días* y *entregas al cliente antes que al transportista*: son pocos casos y muy probablemente errores de registro o pedidos extraviados. No se eliminan porque un retraso extremo es justo el tipo de pedido que produce malas reseñas; XGBoost parte por umbrales, así que un valor extremo no distorsiona el modelo como lo haría en una regresión.
- *`cuotas_max == 0`*: valor imposible (mínimo 1 cuota), solo un par de casos; se deja, porque su efecto es nulo.
- *Divisiones por cero*: `precio_por_item` reemplaza `n_items = 0` por `NaN`, `flete_relativo` divide entre `precio_total + 1`, y los infinitos se convierten a `NaN` antes de imputar.
- *Fechas*: `preparacion.py` convierte todo a fecha con `errors="coerce"` (un texto mal escrito queda `NaT` y no rompe la API) y quita la zona horaria, para que funcione igual con fechas de la base, de la API o del calificador.

### Diccionario de variables (`preparacion.py`)

Todas se calculan **solo** con las 17 columnas de entrada del contrato y con información conocida **en el momento de la entrega**. Nada de la reseña entra al modelo.

| Variable | Cómo se calcula | Qué significa para el negocio |
|---|---|---|
| `dias_retraso` | `order_delivered_customer_date − order_estimated_delivery_date` (días, con decimales) | Cuánto se incumplió (positivo) o se adelantó (negativo) la fecha prometida. Es la promesa central de Olist al cliente. |
| `retrasado` | 1 si `dias_retraso > 0`, si no 0 | Incumplió la promesa de entrega, sin importar por cuánto. |
| `dias_entrega_total` | entrega al cliente − compra | Espera total que vivió el cliente. |
| `dias_aprobacion` | aprobación del pago − compra | Fricción en el pago (antifraude, boleto que tarda en pagarse). |
| `dias_hasta_transportista` | entrega al transportista − compra | Rapidez del vendedor para despachar (responsabilidad del vendedor). |
| `dias_transporte` | entrega al cliente − entrega al transportista | Tiempo en manos del transportista (responsabilidad logística). |
| `dias_prometidos` | fecha prometida − compra | Qué tan larga fue la promesa: promesas largas suelen ser envíos lejanos o complejos. |
| `dia_semana_compra` | día de la semana de la compra (0 = lunes) | Pedidos de fin de semana arrancan su proceso más tarde. |
| `n_items` | del contrato: número de artículos | Pedidos con más artículos tienen más puntos de falla (faltantes, envíos parciales). |
| `n_vendedores` | del contrato: vendedores distintos | Varios vendedores = varios paquetes y llegadas en fechas distintas. |
| `precio_total` | del contrato: suma de `price` | Valor del pedido: a mayor gasto, mayor expectativa. |
| `flete_total` | del contrato: suma de `freight_value` | Costo de envío pagado. |
| `peso_total_g` | del contrato: suma del peso | Paquetes pesados o voluminosos son más difíciles de transportar. |
| `fotos_promedio` | del contrato: promedio de fotos | Proxy de calidad del anuncio: menos fotos → más riesgo de "no era lo que esperaba". |
| `cuotas_max` | del contrato: máximo de cuotas | Compromiso financiero del cliente. |
| `precio_por_item` | `precio_total / n_items` | Ticket promedio por artículo. |
| `flete_relativo` | `flete_total / (precio_total + 1)` | Qué proporción del precio fue envío: un flete caro relativo eleva la expectativa. |
| `mismo_estado` | 1 si `estado_cliente == estado_vendedor` | Envío local (más corto y confiable) vs. interestatal. |
| `estado_cliente` | del contrato (categórica, one-hot) | Geografía del cliente: regiones lejanas (Norte/Nordeste) tienen logística más difícil. |
| `estado_vendedor` | del contrato (categórica, one-hot) | Origen del envío. |
| `categoria` | del contrato (categórica, one-hot; faltante → `desconocido`) | Algunas categorías (muebles, cama/mesa/baño) generan más quejas. |
| `tipo_pago` | del contrato (categórica, one-hot) | Medio de pago principal (boleto implica aprobación más lenta). |

**Variable que quitamos:** `mes_compra`. El modelo se entrena hasta mayo de 2018 y se califica con junio–agosto de 2018: un mes del año calendario no repite su contexto de un año a otro (huelgas, Black Friday, temporadas de la logística), así que la variable aprendería eventos del pasado que no se repetirán. Las categorías poco frecuentes (menos de 30 pedidos) se agrupan en una categoría "rara" (`min_frequency=30`) para no sobreajustar.

## Parte 2 · Modelo de riesgo

Recuerden:

- La validación es **por fecha**.
- Comparen contra la regla "ordenar por días de retraso" usando **precisión en el 10% de mayor riesgo**.
- Las funciones que transforman las columnas del contrato van en `preparacion.py` y se importan aquí.

In [ ]:
%%writefile preparacion.py
"""
preparacion.py — Variables del modelo de riesgo de mala reseña (Proyecto final Olist).

Convierte las 17 columnas de entrada del contrato en las variables que usa el modelo.
El pipeline guardado en modelo_riesgo.joblib llama a `crear_variables` dentro de un
FunctionTransformer, por eso este archivo debe estar junto al .joblib (notebook y API).

Regla de oro (sin fuga de datos): solo se usa información conocida en el momento en que
el pedido se ENTREGA. Nada de la reseña (texto, largo, fechas de la reseña) entra aquí.
"""
import numpy as np
import pandas as pd

FECHAS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

NUMEROS_CONTRATO = ["n_items", "n_vendedores", "precio_total", "flete_total",
                    "peso_total_g", "fotos_promedio", "cuotas_max"]

NUMERICAS = [
    # Tiempos de la entrega (lo que más explica una mala reseña)
    "dias_retraso", "retrasado", "dias_entrega_total", "dias_aprobacion",
    "dias_hasta_transportista", "dias_transporte", "dias_prometidos",
    # Calendario (sin el mes: no generaliza a meses que el modelo no ha visto)
    "dia_semana_compra",
    # Contenido del pedido
    "n_items", "n_vendedores", "precio_total", "flete_total", "peso_total_g",
    "fotos_promedio", "cuotas_max",
    # Derivadas
    "precio_por_item", "flete_relativo", "mismo_estado",
]

CATEGORICAS = ["estado_cliente", "estado_vendedor", "categoria", "tipo_pago"]


def _a_fecha(serie):
    """Convierte texto o fechas a datetime sin zona horaria; lo que no se entiende queda NaT."""
    fechas = pd.to_datetime(serie, errors="coerce", utc=True, format="mixed")
    return fechas.dt.tz_convert(None)


def _dias(fin, inicio):
    return (fin - inicio).dt.total_seconds() / 86400


def crear_variables(df):
    """Recibe un DataFrame con las columnas del contrato y devuelve las variables del modelo."""
    df = pd.DataFrame(df).copy()

    # Tolerancia: si falta una columna opcional, se crea vacía (el imputador la rellena).
    for col in FECHAS + NUMEROS_CONTRATO + ["estado_cliente", "estado_vendedor", "categoria", "tipo_pago"]:
        if col not in df.columns:
            df[col] = np.nan

    for col in FECHAS:
        df[col] = _a_fecha(df[col])
    for col in NUMEROS_CONTRATO:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    compra = df["order_purchase_timestamp"]
    aprobado = df["order_approved_at"]
    transportista = df["order_delivered_carrier_date"]
    entrega = df["order_delivered_customer_date"]
    prometida = df["order_estimated_delivery_date"]

    X = pd.DataFrame(index=df.index)
    X["dias_retraso"] = _dias(entrega, prometida)
    X["retrasado"] = (X["dias_retraso"] > 0).astype(int)
    X["dias_entrega_total"] = _dias(entrega, compra)
    X["dias_aprobacion"] = _dias(aprobado, compra)
    X["dias_hasta_transportista"] = _dias(transportista, compra)
    X["dias_transporte"] = _dias(entrega, transportista)
    X["dias_prometidos"] = _dias(prometida, compra)
    X["dia_semana_compra"] = compra.dt.dayofweek

    for col in NUMEROS_CONTRATO:
        X[col] = df[col]

    X["precio_por_item"] = df["precio_total"] / df["n_items"].replace(0, np.nan)
    X["flete_relativo"] = df["flete_total"] / (df["precio_total"] + 1)
    X["mismo_estado"] = (df["estado_cliente"] == df["estado_vendedor"]).astype(int)

    for col in CATEGORICAS:
        X[col] = df[col].fillna("desconocido").astype(str)

    X = X.replace([np.inf, -np.inf], np.nan)
    return X[NUMERICAS + CATEGORICAS]

In [ ]:
from preparacion import crear_variables      # su archivo, en la misma carpeta

In [ ]:
import json
import joblib
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from preparacion import NUMERICAS, CATEGORICAS

ENTRADAS = CONTRATO[:-1]          # las 17 columnas de entrada (todas menos mala_resena)


def precision_10(y, probabilidades):
    """Proporción de malas reseñas entre el 10% de pedidos con mayor probabilidad."""
    y = np.asarray(y)
    k = int(len(y) * 0.10)
    orden = np.argsort(-np.asarray(probabilidades))
    return y[orden[:k]].mean()


def dias_de_retraso(df):
    """La 'regla del retraso': ordena por días de retraso contra la fecha prometida, sin modelo."""
    return ((df["order_delivered_customer_date"] - df["order_estimated_delivery_date"])
            .dt.total_seconds() / 86400).fillna(0)


def construir_pipeline(modelo, escalar=False):
    pasos_num = [SimpleImputer(strategy="median")]
    if escalar:
        pasos_num.append(StandardScaler())
    preprocesamiento = ColumnTransformer([
        ("num", Pipeline([(f"p{i}", p) for i, p in enumerate(pasos_num)]), NUMERICAS),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), CATEGORICAS),
    ])
    return Pipeline([
        ("variables", FunctionTransformer(crear_variables)),
        ("prep", preprocesamiento),
        ("modelo", modelo),
    ])


def nuevos_modelos():
    return {
        "Regresión logística": construir_pipeline(LogisticRegression(max_iter=2000), escalar=True),
        "XGBoost": construir_pipeline(XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05, min_child_weight=20,
            subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42)),
    }


def evaluar(entrenamiento, validacion):
    """Entrena los modelos y los compara con el azar y la regla del retraso en el periodo de validación."""
    y_val = validacion["mala_resena"]
    modelos = nuevos_modelos()
    filas = [
        {"modelo": "Azar (tasa general)", "precision_10": y_val.mean(), "auc": np.nan},
        {"modelo": "Regla: días de retraso", "precision_10": precision_10(y_val, dias_de_retraso(validacion)),
         "auc": roc_auc_score(y_val, dias_de_retraso(validacion))},
    ]
    probas = {}
    for nombre, pipe in modelos.items():
        pipe.fit(entrenamiento[ENTRADAS], entrenamiento["mala_resena"])
        proba = pipe.predict_proba(validacion[ENTRADAS])[:, 1]
        probas[nombre] = proba
        filas.append({"modelo": nombre, "precision_10": precision_10(y_val, proba), "auc": roc_auc_score(y_val, proba)})
    return pd.DataFrame(filas).round(3), modelos, probas


# ---- Validación principal por fecha: se entrena con el pasado y se valida con lo más reciente ----
pedidos = pedidos.sort_values("order_purchase_timestamp").reset_index(drop=True)
CORTE = pd.Timestamp("2018-03-01")
entrenamiento = pedidos[pedidos["order_purchase_timestamp"] < CORTE]
validacion = pedidos[pedidos["order_purchase_timestamp"] >= CORTE]
y_val = validacion["mala_resena"]

print(f"Entrenamiento: {len(entrenamiento)} pedidos ({entrenamiento['order_purchase_timestamp'].min().date()} a "
      f"{entrenamiento['order_purchase_timestamp'].max().date()}) | tasa de mala reseña: {entrenamiento['mala_resena'].mean():.3f}")
print(f"Validación:    {len(validacion)} pedidos ({validacion['order_purchase_timestamp'].min().date()} a "
      f"{validacion['order_purchase_timestamp'].max().date()}) | tasa de mala reseña: {y_val.mean():.3f}")

comparacion, modelos, probas_val = evaluar(entrenamiento, validacion)
comparacion

In [ ]:
## Variables más importantes (XGBoost entrenado en el periodo de entrenamiento)
pipe_xgb = modelos["XGBoost"]
nombres = pipe_xgb.named_steps["prep"].get_feature_names_out()
importancia = pd.Series(pipe_xgb.named_steps["modelo"].feature_importances_, index=nombres).sort_values(ascending=False)
importancia.index = [n.split("__", 1)[1] for n in importancia.index]

importancia.head(10).sort_values().plot.barh(figsize=(7, 4), title="Top 10 variables (importancia en XGBoost)")
plt.tight_layout()
plt.show()

# Los coeficientes de la regresión logística confirman la dirección del efecto
pipe_lr = modelos["Regresión logística"]
coef = pd.Series(pipe_lr.named_steps["modelo"].coef_[0],
                 index=[n.split("__", 1)[1] for n in pipe_lr.named_steps["prep"].get_feature_names_out()])
print("Coeficientes (variables escaladas) de las 3 variables más importantes en XGBoost:")
print(coef.reindex(importancia.index[:3]).round(3))
importancia.head(10).round(3)

In [ ]:
## Tasas por mes: contexto para entender la validación y el periodo oculto
mes = pedidos["order_purchase_timestamp"].dt.to_period("M")
llego_tarde = (pedidos["order_delivered_customer_date"] > pedidos["order_estimated_delivery_date"]).astype(int)
por_mes = pd.DataFrame({
    "pedidos": pedidos.groupby(mes)["order_id"].count(),
    "tasa_retraso": llego_tarde.groupby(mes).mean(),
    "tasa_mala_resena": pedidos.groupby(mes)["mala_resena"].mean(),
})
print("Tasa de mala reseña entre pedidos retrasados:", round(pedidos.loc[llego_tarde == 1, "mala_resena"].mean(), 3))
print("Tasa de mala reseña entre pedidos a tiempo:  ", round(pedidos.loc[llego_tarde == 0, "mala_resena"].mean(), 3))
por_mes.round(3).tail(12)

In [ ]:
## Prueba de robustez: ¿qué pasa en un periodo con pocos retrasos?
# Abril y mayo de 2018 tienen muy pocos retrasos (ver tabla anterior), como podría ocurrir en junio-agosto.
# Entrenamos hasta marzo y validamos solo con abril-mayo.
CORTE_2 = pd.Timestamp("2018-04-01")
entrenamiento_2 = pedidos[pedidos["order_purchase_timestamp"] < CORTE_2]
validacion_2 = pedidos[pedidos["order_purchase_timestamp"] >= CORTE_2]
retraso_val_2 = (validacion_2["order_delivered_customer_date"] > validacion_2["order_estimated_delivery_date"]).mean()
print(f"Validación abril-mayo: {len(validacion_2)} pedidos | tasa de retraso: {retraso_val_2:.3f} | "
      f"tasa de mala reseña: {validacion_2['mala_resena'].mean():.3f}")
comparacion_2, _, _ = evaluar(entrenamiento_2, validacion_2)
comparacion_2

### Modelo final y prueba del contrato

Entrenen el modelo final con todos los pedidos, guárdenlo como `modelo_riesgo.joblib` y comprueben que, **cargado desde el archivo**, predice con filas que solo traen las columnas del contrato:

In [ ]:
modelos_ok = comparacion[~comparacion["modelo"].isin(["Azar (tasa general)", "Regla: días de retraso"])]
mejor = modelos_ok.loc[modelos_ok["precision_10"].idxmax(), "modelo"]
print("Modelo elegido (mayor precisión en el 10% en la validación por fecha):", mejor)

modelo_final = clone(nuevos_modelos()[mejor])
modelo_final.fit(pedidos[ENTRADAS], pedidos["mala_resena"])
joblib.dump(modelo_final, "modelo_riesgo.joblib")

# Umbral de "riesgo alto": la probabilidad que separa al 10% más riesgoso de los datos de entrenamiento
proba_todos = modelo_final.predict_proba(pedidos[ENTRADAS])[:, 1]
umbral = float(np.quantile(proba_todos, 0.90))
with open("umbral.json", "w") as f:
    json.dump({"umbral": umbral, "descripcion": "percentil 90 de la probabilidad en los pedidos de entrenamiento"}, f)
print("Umbral del 10% de mayor riesgo:", round(umbral, 4))

In [ ]:
import joblib
cargado = joblib.load("modelo_riesgo.joblib")
cargado.predict_proba(pedidos[CONTRATO[:-1]].head(5))[:, 1]

In [ ]:
# Prueba extra del contrato: filas "como las enviaría el calificador" (fechas como texto y faltantes)
prueba = pedidos[CONTRATO].head(3).copy()
for c in FECHAS:
    prueba[c] = prueba[c].astype(str)
prueba.loc[prueba.index[0], ["order_approved_at", "categoria", "peso_total_g"]] = None
print(cargado.predict_proba(prueba).round(4)[:, 1], "<- también funciona con la columna mala_resena incluida y con faltantes")

### Preguntas (respóndanlas en Markdown, con cifras de sus salidas)

1. ¿Cuánto mejora su modelo sobre la regla de retraso? ¿Y sobre el azar?
2. ¿Qué tres variables pesan más y por qué tiene sentido para el negocio?
3. ¿Qué pasaría con su modelo en un periodo con muy pocos retrasos?

In [ ]:
from IPython.display import Markdown, display

fila = comparacion.set_index("modelo")
p_mod, p_regla, p_azar = fila.loc[mejor, "precision_10"], fila.loc["Regla: días de retraso", "precision_10"], fila.loc["Azar (tasa general)", "precision_10"]
fila2 = comparacion_2.set_index("modelo")
p2_mod, p2_regla, p2_azar = fila2.loc[mejor, "precision_10"], fila2.loc["Regla: días de retraso", "precision_10"], fila2.loc["Azar (tasa general)", "precision_10"]
top3 = list(importancia.index[:3])

respuestas_p2 = f"""
**1. ¿Cuánto mejora el modelo sobre la regla del retraso y sobre el azar?** (validación por fecha: {len(validacion):,} pedidos desde {CORTE.date()})

| | Precisión en el 10% | AUC |
|---|---|---|
| Azar (tasa general) | {p_azar:.3f} | — |
| Regla: días de retraso | {p_regla:.3f} | {fila.loc['Regla: días de retraso','auc']:.3f} |
| Regresión logística | {fila.loc['Regresión logística','precision_10']:.3f} | {fila.loc['Regresión logística','auc']:.3f} |
| XGBoost | {fila.loc['XGBoost','precision_10']:.3f} | {fila.loc['XGBoost','auc']:.3f} |

- Con el modelo elegido (**{mejor}**), de cada 10 clientes que el equipo llamaría, unos **{p_mod*10:.1f}** iban a dejar una mala reseña.
- Frente al **azar**: {p_mod:.3f} vs {p_azar:.3f} → **+{(p_mod-p_azar)*100:.1f} puntos** ({p_mod/p_azar:.2f} veces).
- Frente a la **regla del retraso**: {p_mod:.3f} vs {p_regla:.3f} → **+{(p_mod-p_regla)*100:.1f} puntos** ({p_mod/p_regla:.2f} veces). En este periodo hubo muchos retrasos, así que la regla ya es fuerte y la ventaja del modelo es moderada.

**2. Las tres variables que más pesan en XGBoost:** `{top3[0]}`, `{top3[1]}` y `{top3[2]}` (importancias {importancia.iloc[0]:.3f}, {importancia.iloc[1]:.3f} y {importancia.iloc[2]:.3f}).

**3. ¿Qué pasaría en un periodo con muy pocos retrasos?** Lo medimos: abril–mayo de 2018 tuvo una tasa de retraso de **{retraso_val_2:.3f}** (vs. {por_mes['tasa_retraso'].loc['2018-02':'2018-03'].mean():.3f} en feb–mar).
Entrenando hasta marzo y validando en abril–mayo:
- Modelo: **{p2_mod:.3f}** · regla del retraso: **{p2_regla:.3f}** · azar: **{p2_azar:.3f}**.
- La precisión del modelo cae de {p_mod:.3f} a {p2_mod:.3f}, pero la regla cae mucho más (de {p_regla:.3f} a {p2_regla:.3f}): la ventaja sobre la regla pasa de +{(p_mod-p_regla)*100:.1f} a **+{(p2_mod-p2_regla)*100:.1f} puntos**.
"""
display(Markdown(respuestas_p2))

**Por qué tienen sentido esas variables para el negocio.** Las variables de retraso (`retrasado`, `dias_retraso`) miden si Olist cumplió la promesa de entrega: es la experiencia más visible para el cliente, y un pedido que llega tarde tiene varias veces más probabilidad de recibir 1–2 estrellas que uno a tiempo (ver la celda de tasas por mes). Las variables de tamaño del pedido (`n_items`, `n_vendedores`) capturan otra fuente de quejas: los pedidos con varios artículos o vendedores llegan en varios paquetes, en fechas distintas o incompletos, y el cliente reseña cuando le falta algo. La regresión logística confirma la dirección: `retrasado` y `n_items` tienen coeficiente positivo (más riesgo). El coeficiente de `dias_retraso` sale casi nulo porque, una vez que el modelo sabe *si* hubo retraso, los días extra aportan poca información lineal adicional; XGBoost sí aprovecha esos días porque puede cortar en tramos (por ejemplo, más de una semana de retraso).

**Interpretación de la pregunta 3.** Cuando casi no hay retrasos, la señal más fuerte casi desaparece: menos pedidos "obvios" para llamar, y el 10% de arriba se llena con pedidos de riesgo moderado, así que la precisión baja. Aun así, el modelo resiste mejor que la regla porque, además del retraso, usa el tamaño del pedido, el número de vendedores y los tiempos de entrega y despacho (las siguientes variables en importancia). Por eso esperamos que en junio–agosto de 2018 (periodo oculto) la precisión sea **menor** que en nuestra validación principal, pero aun claramente por encima de la regla. Recomendación: recalcular el umbral y reentrenar periódicamente, porque la tasa base de malas reseñas cambia con el contexto logístico.

## Parte 3 · Pregunta abierta

Opción elegida: **3B · Segmentación de vendedores**

Justifiquen la elección y cada decisión. Si eligen 3A, guarden `pronostico_junio.csv` con columnas `fecha, pronostico, bajo, alto` (30 filas, del 1 al 30 de junio de 2018).

**Por qué 3B.** El modelo de riesgo de la Parte 2 dice *a qué clientes llamar*; la segmentación dice *con qué vendedores trabajar y cómo*. Es la pieza que ataca la causa: si un grupo de vendedores concentra retrasos y malas reseñas, Olist puede actuar sobre ellos (acompañamiento, reglas de despacho) antes de que haya más pedidos que llamar. Un pronóstico de volumen (3A) es útil para planificar capacidad, pero no cambia la experiencia del cliente.

In [ ]:
# Una fila por vendedor con al menos 10 pedidos entregados. Las variables se calculan con SQL.
CONSULTA_VENDEDORES = """
WITH resena AS (
    SELECT DISTINCT ON (order_id) order_id, review_score
    FROM final.order_reviews
    ORDER BY order_id, review_answer_timestamp DESC NULLS LAST
)
SELECT oi.seller_id,
       MIN(s.seller_state)                                                   AS seller_state,
       COUNT(DISTINCT oi.order_id)                                           AS n_pedidos,
       SUM(oi.price)::float                                                  AS ventas_total,
       AVG(oi.price)::float                                                  AS precio_promedio,
       (SUM(oi.freight_value) / NULLIF(SUM(oi.price), 0))::float             AS flete_relativo,
       AVG(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1.0 ELSE 0.0 END)::float                                AS tasa_retraso,
       COUNT(DISTINCT p.product_category_name)                               AS n_categorias,
       AVG(CASE WHEN r.review_score <= 2 THEN 1.0
                WHEN r.review_score IS NOT NULL THEN 0.0 END)::float        AS tasa_mala_resena
FROM final.order_items oi
JOIN final.orders o       ON o.order_id  = oi.order_id
JOIN final.sellers s      ON s.seller_id = oi.seller_id
LEFT JOIN final.products p ON p.product_id = oi.product_id
LEFT JOIN resena r        ON r.order_id  = oi.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY oi.seller_id
HAVING COUNT(DISTINCT oi.order_id) >= 10
ORDER BY oi.seller_id
"""
vendedores = pd.read_sql(CONSULTA_VENDEDORES, motor)
print(vendedores.shape)
vendedores.describe().T.round(2)

**Variables y preparación.**
- Para agrupar usamos 6 variables de comportamiento: `n_pedidos` (tamaño), `ventas_total` (facturación), `precio_promedio` (ticket), `flete_relativo` (peso del envío en el precio), `tasa_retraso` (cumplimiento) y `n_categorias` (diversificación).
- `tasa_mala_resena` **no** se usa para agrupar: la guardamos para describir los grupos después y comprobar si la segmentación tiene relación con la reputación.
- Las variables de volumen y precio tienen colas largas (pocos vendedores muy grandes), así que les aplicamos `log1p` antes de escalar. Después las estandarizamos (`StandardScaler`), porque K-Means usa distancias y sin escalar dominarían las ventas en reales.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

VARS = ["n_pedidos", "ventas_total", "precio_promedio", "flete_relativo", "tasa_retraso", "n_categorias"]
LOG = ["n_pedidos", "ventas_total", "precio_promedio", "flete_relativo", "n_categorias"]
print("Asimetría antes del logaritmo:", vendedores[LOG].skew().round(1).to_dict())

X = vendedores[VARS].copy()
for col in LOG:
    X[col] = np.log1p(X[col])
X = X.fillna(X.median())
X_escalado = StandardScaler().fit_transform(X)

ks = range(2, 9)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_escalado)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_escalado, km.labels_))

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5))
ejes[0].plot(list(ks), inercias, marker="o"); ejes[0].set_title("Codo (inercia)"); ejes[0].set_xlabel("k")
ejes[1].plot(list(ks), siluetas, marker="o"); ejes[1].set_title("Silueta"); ejes[1].set_xlabel("k")
plt.tight_layout()
plt.show()
tabla_k = pd.DataFrame({"k": list(ks), "inercia": np.round(inercias, 1), "silueta": np.round(siluetas, 3)})
tabla_k

In [ ]:
K = 4
s = dict(zip(tabla_k["k"], tabla_k["silueta"]))
display(Markdown(f"""
**Elección de k = {K}.**
- *Silueta:* k=2 → {s[2]:.3f}, k=3 → {s[3]:.3f}, k=4 → {s[4]:.3f}, k=5 → {s[5]:.3f}. Hasta k=4 la silueta es prácticamente igual (diferencia máxima {max(s[2], s[3], s[4]) - min(s[2], s[3], s[4]):.3f}) y cae en k=5.
- *Codo:* la inercia baja de {tabla_k.inercia.iloc[0]:,.0f} (k=2) a {tabla_k.inercia.iloc[2]:,.0f} (k=4); a partir de ahí cada grupo extra reduce menos.
- k=2 tiene la silueta más alta, pero por muy poco; con un empate práctico decide el negocio.
- *Criterio de negocio:* con 2 o 3 grupos se mezclan perfiles que piden acciones distintas (por ejemplo, el vendedor con retrasos queda junto al vendedor grande). Con 4 cada grupo tiene una acción clara y ninguno es demasiado pequeño para gestionarlo. Por eso elegimos k=4: la mayor granularidad útil sin perder calidad de agrupamiento.
- La silueta es baja en todos los k (los vendedores forman un continuo, no "islas"), así que los segmentos son una **herramienta de gestión**, no categorías naturales.
"""))

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X_escalado)
vendedores["segmento"] = kmeans.labels_

perfil = vendedores.groupby("segmento")[VARS + ["tasa_mala_resena"]].median().round(3)
perfil["n_vendedores"] = vendedores["segmento"].value_counts().sort_index()
perfil["ventas_del_grupo_%"] = (vendedores.groupby("segmento")["ventas_total"].sum() / vendedores["ventas_total"].sum() * 100).round(1)

# Nombre por perfil, asignado con reglas sobre las medianas (no depende del número que K-Means le ponga al grupo)
nombres, restantes = {}, list(perfil.index)
seg = perfil.loc[restantes, "tasa_retraso"].idxmax();   nombres[seg] = "Con problemas de entrega"; restantes.remove(seg)
seg = perfil.loc[restantes, "n_pedidos"].idxmax();      nombres[seg] = "Grandes y diversificados"; restantes.remove(seg)
seg = perfil.loc[restantes, "flete_relativo"].idxmax(); nombres[seg] = "Ticket bajo, flete caro";  restantes.remove(seg)
seg = restantes[0];                                     nombres[seg] = "Ticket alto y entrega confiable"

ACCIONES = {
    "Con problemas de entrega": "Plan de mejora logística: alerta temprana de despacho, revisión de su fecha prometida y prioridad de sus pedidos en el modelo de riesgo; si no mejora, limitar su visibilidad.",
    "Grandes y diversificados": "Gestor de cuenta dedicado: son el motor de ventas; proteger su operación con monitoreo semanal de retrasos y reseñas.",
    "Ticket bajo, flete caro": "Programa de envíos: tarifas de flete negociadas, incentivo a combos/pedido mínimo para que el flete pese menos en el precio.",
    "Ticket alto y entrega confiable": "Crecimiento: más exposición en el sitio y campañas, porque entregan bien y venden caro; ayudarles a ampliar catálogo.",
}
perfil.insert(0, "nombre", perfil.index.map(nombres))
perfil["accion"] = perfil["nombre"].map(ACCIONES)
vendedores["segmento_nombre"] = vendedores["segmento"].map(nombres)
print(f"Tasa de mala reseña general de los vendedores segmentados (mediana): {vendedores['tasa_mala_resena'].median():.3f}")
perfil

In [ ]:
pca = PCA(n_components=2, random_state=42)
componentes = pca.fit_transform(X_escalado)
vendedores["pca1"], vendedores["pca2"] = componentes[:, 0], componentes[:, 1]

plt.figure(figsize=(8, 5.5))
for seg_id, nombre in nombres.items():
    marca = vendedores["segmento"] == seg_id
    plt.scatter(componentes[marca, 0], componentes[marca, 1], s=12, alpha=0.7, label=f"{nombre} ({marca.sum()})")
plt.xlabel(f"Componente 1 ({pca.explained_variance_ratio_[0]*100:.0f}% de la varianza)")
plt.ylabel(f"Componente 2 ({pca.explained_variance_ratio_[1]*100:.0f}% de la varianza)")
plt.title("Segmentos de vendedores (PCA)")
plt.legend()
plt.tight_layout()
plt.show()

cargas = pd.DataFrame(pca.components_.T, index=VARS, columns=["PC1", "PC2"]).round(2)
print("Cargas de cada variable en los componentes:")
print(cargas)
vendedores.to_csv("segmentos_vendedores.csv", index=False)
perfil.to_csv("perfil_segmentos.csv")

In [ ]:
p = perfil.set_index("nombre")
display(Markdown(f"""
**Perfil y acción por segmento** (medianas de cada grupo)

| Segmento | Vendedores | Pedidos | Ticket (R$) | Flete relativo | Tasa de retraso | Mala reseña | Acción |
|---|---|---|---|---|---|---|---|
""" + "\n".join(
    f"| **{n}** | {int(r.n_vendedores)} | {r.n_pedidos:.0f} | {r.precio_promedio:.0f} | {r.flete_relativo:.2f} | {r.tasa_retraso:.3f} | {r.tasa_mala_resena:.3f} | {ACCIONES[n]} |"
    for n, r in p.iterrows()) + f"""

- El grupo **Con problemas de entrega** tiene la mayor tasa de retraso y también la mayor tasa de mala reseña ({p.loc['Con problemas de entrega','tasa_mala_resena']:.3f}), aunque esa variable no se usó para agrupar: confirma que la segmentación está conectada con la reputación.
- **Grandes y diversificados** concentra el {p.loc['Grandes y diversificados','ventas_del_grupo_%']:.1f}% de las ventas de los vendedores segmentados.
"""))

## Parte 4 · Deep learning: sentimiento de los comentarios

Clasifiquen una muestra de comentarios con un modelo preentrenado de Hugging Face que entienda portugués y compárenlo con las estrellas. Activen la GPU de Colab para que sea rápido.

**Modelo:** `cardiffnlp/twitter-xlm-roberta-base-sentiment` (XLM-RoBERTa entrenado en 8 idiomas, portugués incluido) que devuelve `positive`, `neutral` o `negative`: las mismas tres clases que podemos formar con las estrellas.

In [ ]:
!pip install -q transformers sentencepiece

In [ ]:
import torch
from transformers import pipeline

dispositivo = 0 if torch.cuda.is_available() else -1
print("Usando", "GPU" if dispositivo == 0 else "CPU")
clasificador = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
                        device=dispositivo)

# Muestra de 1,000 comentarios con texto, con semilla fija (ORDER BY para que la consulta sea reproducible)
comentarios = pd.read_sql("""
    SELECT review_id, order_id, review_score, review_comment_message
    FROM final.order_reviews
    WHERE review_comment_message IS NOT NULL AND TRIM(review_comment_message) <> ''
    ORDER BY review_id, order_id
""", motor)
print("Comentarios con texto disponibles:", len(comentarios))
muestra = comentarios.sample(1000, random_state=42).reset_index(drop=True)
muestra["review_score"] = muestra["review_score"].astype(int)

resultados_sent = clasificador(muestra["review_comment_message"].tolist(), batch_size=32, truncation=True)
muestra["sentimiento"] = [r["label"].lower() for r in resultados_sent]
muestra["confianza"] = [r["score"] for r in resultados_sent]

# Estrellas agrupadas igual que el modelo: 1-2 negativo, 3 neutro, 4-5 positivo
def grupo_estrellas(e):
    return "negative" if e <= 2 else "neutral" if e == 3 else "positive"

ORDEN = ["negative", "neutral", "positive"]
muestra["estrellas_grupo"] = muestra["review_score"].apply(grupo_estrellas)
muestra["coincide"] = (muestra["estrellas_grupo"] == muestra["sentimiento"]).astype(int)

tabla = pd.crosstab(muestra["estrellas_grupo"], muestra["sentimiento"]).reindex(index=ORDEN, columns=ORDEN, fill_value=0)
tabla.index.name, tabla.columns.name = "estrellas", "sentimiento del modelo"
pct = (tabla.div(tabla.sum(axis=1), axis=0) * 100).round(1)
coincidencia = muestra["coincide"].mean()

print("Tabla cruzada (filas = estrellas, columnas = sentimiento del modelo):")
print(tabla)
print("\nMismo cuadro en % por fila:")
print(pct)
print(f"\nCoincidencia entre estrellas y sentimiento: {coincidencia * 100:.1f}%")

muestra.to_csv("sentimiento_muestra.csv", index=False)
muestra.head()

### Preguntas

1. ¿En qué porcentaje coincide el sentimiento con las estrellas? ¿Dónde está la mayor confusión?
2. Muestren al menos 5 desacuerdos y clasifíquenlos: ¿error del modelo o del cliente?
3. ¿Qué dice el texto que las estrellas no dicen?

In [ ]:
# Pregunta 1: confusiones (celdas fuera de la diagonal)
confusiones = pd.DataFrame({"comentarios": tabla.stack(), "% de su fila": pct.stack()})
confusiones = confusiones[confusiones.index.get_level_values(0) != confusiones.index.get_level_values(1)]
confusiones = confusiones.sort_values("comentarios", ascending=False)
mayor = confusiones.index[0]
fila_peor = pct.values.diagonal().argmin()
confusiones

In [ ]:
# Pregunta 2: desacuerdos para leer (muestra aleatoria de cada tipo)
pd.set_option("display.max_colwidth", 300)

def tomar(filtro, n):
    sub = muestra[filtro]
    return sub.sample(min(n, len(sub)), random_state=1)

casos = pd.concat([
    tomar((muestra["review_score"] >= 4) & (muestra["sentimiento"] == "negative"), 4),   # buenas estrellas, texto negativo
    tomar((muestra["review_score"] <= 2) & (muestra["sentimiento"] == "positive"), 2),   # malas estrellas, texto positivo
    tomar((muestra["review_score"] <= 2) & (muestra["sentimiento"] == "neutral"), 4),    # malas estrellas, texto neutro
    tomar((muestra["review_score"] == 3) & (muestra["sentimiento"] != "neutral"), 2),    # 3 estrellas, texto no neutro
])
print("Total de desacuerdos en la muestra:", int((muestra["coincide"] == 0).sum()))
casos[["review_score", "sentimiento", "confianza", "review_comment_message"]]

In [ ]:
# (a) Desacuerdos que leímos y clasificamos en nuestra primera corrida con el modelo real.
#     Los buscamos por su texto para que la tabla sea reproducible aunque cambie la muestra.
LEIDOS = [
    ("ainda não recebi o produto, não consta nem a chegada",
     "Error del modelo", "Queja grave (no llegó el producto) escrita en tono informativo, sin palabras emocionales: el modelo la ve neutral."),
    ("bolas de bilhar que comprei é totalmente diferente",
     "Error del modelo", "Producto distinto al anunciado y pregunta por el reembolso: claramente negativo, pero formulado como pregunta."),
    ("Capinha e muito boa, mas a película e horrível",
     "Texto mixto", "Elogia un producto y critica otro; el cliente pondera y pone 4, el modelo se fija en 'horrível'."),
    ("dificuldade na entrega",
     "Error del modelo", "Relato de mala atención y entrega sin fecha, descrito de forma narrativa: el modelo no capta la queja."),
    ("não deveria mais trabalhar com os correios",
     "Error del cliente", "Queja directa contra el servicio de entrega, pero el cliente pone 3 estrellas (suaviza su nota)."),
    ("Os correios atrasaram",
     "Error del cliente", "Se queja del retraso pero pone 5 estrellas: probablemente califica el producto, no la entrega."),
]
filas = []
for fragmento, quien, por_que in LEIDOS:
    encontrado = pd.read_sql(
        "SELECT review_score, review_comment_message FROM final.order_reviews "
        "WHERE review_comment_message ILIKE %(patron)s ORDER BY review_id LIMIT 1",
        motor, params={"patron": f"%{fragmento}%"})
    if encontrado.empty:
        continue
    texto, estrellas = encontrado.iloc[0]["review_comment_message"], int(encontrado.iloc[0]["review_score"])
    pred = clasificador([texto], truncation=True)[0]
    filas.append({"estrellas": estrellas, "grupo_estrellas": grupo_estrellas(estrellas),
                  "sentimiento_modelo": pred["label"].lower(), "confianza": round(pred["score"], 3),
                  "comentario": texto, "clasificacion": quien, "por_que": por_que})
desacuerdos_clasificados = pd.DataFrame(filas)
desacuerdos_clasificados["es_desacuerdo"] = desacuerdos_clasificados["grupo_estrellas"] != desacuerdos_clasificados["sentimiento_modelo"]
# Si en esta corrida el modelo coincidiera con las estrellas, ese caso ya no es un desacuerdo
desacuerdos_clasificados.loc[~desacuerdos_clasificados["es_desacuerdo"], "clasificacion"] = "Sin desacuerdo en esta corrida"
desacuerdos_clasificados["origen"] = "leído y clasificado a mano"
desacuerdos_clasificados = desacuerdos_clasificados[desacuerdos_clasificados["es_desacuerdo"]].reset_index(drop=True)
desacuerdos_clasificados.to_csv("desacuerdos_clasificados.csv", index=False)
print("Desacuerdos leídos y clasificados:", len(desacuerdos_clasificados))
if len(desacuerdos_clasificados) < 5:
    print("ATENCIÓN: hay menos de 5; leer y clasificar más casos de la tabla `casos` (arriba).")
print(desacuerdos_clasificados["clasificacion"].value_counts().to_string())
desacuerdos_clasificados[["estrellas", "sentimiento_modelo", "confianza", "clasificacion", "comentario", "por_que"]]

In [ ]:
# Pregunta 3: ¿qué temas aparecen en el texto según el grupo de estrellas?
# Patrones revisados para no confundir elogios ("chegou antes do prazo") con quejas
temas = {
    "retraso en la entrega": r"atras|demor|fora do prazo|passou do prazo|depois do prazo|após o prazo|apos o prazo",
    "no llegó el producto": r"não chegou|nao chegou|não recebi|nao recebi|ainda não|ainda nao",
    "defecto o producto dañado": r"defeito|quebrad|danificad|estragad",
    "producto equivocado o distinto": r"errad|diferente|trocad|outro produto",
    "elogio": r"recomendo|ótim|otim|excelente|perfeit|\badorei\b|\badoro\b|antes do prazo",
}
NEGACION_ELOGIO = r"não recomend|nao recomend|não indico|nao indico"


def menciona(textos, tema):
    """True si el comentario menciona el tema (para 'elogio', descarta los 'não recomendo')."""
    t = textos.str.lower()
    m = t.str.contains(temas[tema], regex=True)
    if tema == "elogio":
        m &= ~t.str.contains(NEGACION_ELOGIO, regex=True)
    return m


resumen = pd.DataFrame({
    tema: menciona(muestra["review_comment_message"], tema).groupby(muestra["estrellas_grupo"]).mean() * 100
    for tema in temas
}).reindex(ORDEN).round(1)
resumen.index.name = "estrellas (% de comentarios que mencionan cada tema)"

# El texto también señala problemas que las estrellas esconden: clientes de 4-5 estrellas con texto negativo
ocultos = muestra[(muestra["estrellas_grupo"] == "positive") & (muestra["sentimiento"] == "negative")]
temas_ocultos = pd.Series({t: menciona(ocultos["review_comment_message"], t).mean() * 100 for t in temas}).round(1)
print(f"Comentarios con 4-5 estrellas pero texto negativo: {len(ocultos)}")
print("Temas en esos comentarios (%):", temas_ocultos.to_dict())
resumen.to_csv("temas_comentarios.csv")
resumen

In [ ]:
# Confianza del modelo vs. coincidencia con las estrellas
muestra["intervalo_confianza"] = pd.cut(muestra["confianza"], bins=[0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
analisis_confianza = muestra.groupby("intervalo_confianza", observed=True).agg(
    comentarios=("coincide", "count"), coincidencia=("coincide", "mean")).reset_index()
plt.figure(figsize=(8, 3.5))
plt.bar(analisis_confianza["intervalo_confianza"].astype(str), analisis_confianza["coincidencia"], color="teal", alpha=0.7)
plt.title("Coincidencia con las estrellas según la confianza del modelo")
plt.xlabel("Confianza del modelo"); plt.ylabel("Coincidencia"); plt.ylim(0, 1)
plt.tight_layout(); plt.show()
analisis_confianza.round(3)

In [ ]:
EXPLICA_FILA = {
    "neutral": "Las 3 estrellas son ambiguas: el cliente está \"más o menos\" y su texto suele inclinarse hacia la queja o hacia el elogio.",
    "negative": "Muchas quejas se escriben como relato de hechos (\"no llegó\", \"vino otro producto\"), sin palabras emocionales, y el modelo las lee como neutrales.",
    "positive": "Muchos comentarios positivos son muy cortos o descriptivos (\"recibí el producto\") y el modelo los lee como neutrales.",
}
neg_row, neu_row, pos_row = pct.loc["negative"], pct.loc["neutral"], pct.loc["positive"]
conf_alta = analisis_confianza.iloc[-1]
display(Markdown(f"""
**1. Coincidencia.** El sentimiento del modelo coincide con las estrellas en el **{coincidencia*100:.1f}%** de los 1,000 comentarios.
- Por fila: negativos {neg_row['negative']:.1f}%, neutros {neu_row['neutral']:.1f}%, positivos {pos_row['positive']:.1f}%.
- **Mayor confusión en número de comentarios:** estrellas *{mayor[0]}* que el modelo lee como *{mayor[1]}* ({int(confusiones.iloc[0]['comentarios'])} comentarios, {confusiones.iloc[0]['% de su fila']:.1f}% de su fila).
- **Fila peor clasificada:** *{ORDEN[fila_peor]}* (solo {pct.values.diagonal()[fila_peor]:.1f}% de acierto). {EXPLICA_FILA[ORDEN[fila_peor]]}
- Cuando el modelo tiene confianza {conf_alta['intervalo_confianza']}, coincide en el {conf_alta['coincidencia']*100:.1f}% de los casos ({int(conf_alta['comentarios'])} comentarios).

**2. Desacuerdos clasificados** (tabla de arriba, {len(desacuerdos_clasificados)} desacuerdos leídos y clasificados): {int((desacuerdos_clasificados['clasificacion']=='Error del modelo').sum())} error del modelo, {int((desacuerdos_clasificados['clasificacion']=='Error del cliente').sum())} error (o inconsistencia) del cliente, {int(desacuerdos_clasificados['clasificacion'].str.startswith('Texto mixto').sum())} texto mixto. La tabla `casos` (más arriba) muestra otros desacuerdos de esta muestra para seguir leyendo.

**3. Qué dice el texto que las estrellas no dicen.** Entre los comentarios de 1–2 estrellas, el {resumen.loc['negative','no llegó el producto']:.1f}% menciona que el producto **no llegó**, el {resumen.loc['negative','producto equivocado o distinto']:.1f}% un producto **equivocado o distinto**, el {resumen.loc['negative','retraso en la entrega']:.1f}% **retrasos o plazos** y el {resumen.loc['negative','defecto o producto dañado']:.1f}% **defectos**. Además, hay {len(ocultos)} clientes con 4–5 estrellas cuyo texto es negativo.
"""))

**Conclusión: qué agrega el texto y para qué le sirve a Olist.**
- **Los errores del modelo tienen un patrón** (lo vimos al leer los desacuerdos). Las quejas escritas en tono factual ("todavía no recibí", "llegó otro producto") parecen neutrales para un modelo entrenado en tuits, que busca palabras emocionales. Los errores del cliente también tienen un patrón: califica el producto con muchas estrellas pero se queja de la entrega en el texto.
- **El texto dice el porqué.** Una estrella es un número; el comentario dice si el problema fue el transportista, el vendedor (producto equivocado o defectuoso) o la promesa de plazo. Con eso Olist puede enviar cada queja al equipo responsable (logística, calidad del vendedor, atención) y alimentar la segmentación de vendedores de la Parte 3.
- **Detecta problemas escondidos.** Los clientes que ponen buenas estrellas pero escriben una queja son clientes en riesgo que las estrellas no muestran.
- **Uso práctico:** clasificar automáticamente solo los comentarios en los que el modelo tiene confianza alta y mandar a revisión humana los de baja confianza. El texto **no** se usa en el modelo de riesgo de la Parte 2 (allí sería fuga de datos: el comentario existe solo después de la entrega).

## Extra (opcional, +5) · Red neuronal contra XGBoost

Misma división por fecha y mismas variables. Muestren la curva de pérdida y expliquen el resultado.

In [ ]:
import keras
from keras import layers

keras.utils.set_random_seed(42)

# Mismas variables que XGBoost: el mismo paso de preparación, ahora con escalado y one-hot denso
prep_nn = Pipeline([
    ("variables", FunctionTransformer(crear_variables)),
    ("prep", ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), NUMERICAS),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30, sparse_output=False), CATEGORICAS),
    ])),
])

# Misma división por fecha. Dentro del entrenamiento, el último 15% (por fecha) sirve para el early stopping.
corte_interno = entrenamiento["order_purchase_timestamp"].quantile(0.85)
ent_nn = entrenamiento[entrenamiento["order_purchase_timestamp"] < corte_interno]
val_nn = entrenamiento[entrenamiento["order_purchase_timestamp"] >= corte_interno]

X_ent = prep_nn.fit_transform(ent_nn[ENTRADAS]).astype("float32")
X_valint = prep_nn.transform(val_nn[ENTRADAS]).astype("float32")
X_val = prep_nn.transform(validacion[ENTRADAS]).astype("float32")
y_ent, y_valint = ent_nn["mala_resena"].values, val_nn["mala_resena"].values

red = keras.Sequential([
    keras.Input(shape=(X_ent.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])
red.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy",
            metrics=[keras.metrics.AUC(name="auc")])
parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
historia = red.fit(X_ent, y_ent, validation_data=(X_valint, y_valint), epochs=100, batch_size=256,
                   callbacks=[parada], verbose=0)

perdida, perdida_val = historia.history["loss"], historia.history["val_loss"]
mejor_epoca = int(np.argmin(perdida_val)) + 1
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(perdida) + 1), perdida, label="Entrenamiento")
plt.plot(range(1, len(perdida_val) + 1), perdida_val, label="Validación interna (early stopping)")
plt.axvline(mejor_epoca, color="gray", linestyle="--", label=f"Mejor época ({mejor_epoca})")
plt.title("Curva de pérdida por época · red densa de Keras")
plt.xlabel("Época"); plt.ylabel("Pérdida (entropía cruzada)")
plt.legend(); plt.tight_layout(); plt.show()

proba_nn = red.predict(X_val, verbose=0).ravel()
fila_xgb = comparacion.set_index("modelo").loc["XGBoost"]
comparacion_nn = pd.DataFrame({
    "modelo": ["Red neuronal (Keras)", "XGBoost", "Regla: días de retraso"],
    "precision_10": [precision_10(y_val, proba_nn), fila_xgb["precision_10"], fila.loc["Regla: días de retraso", "precision_10"]],
    "auc": [roc_auc_score(y_val, proba_nn), fila_xgb["auc"], fila.loc["Regla: días de retraso", "auc"]],
}).round(3)
print(f"Filas: entrenamiento de la red {len(ent_nn):,} | validación interna {len(val_nn):,} | validación {len(validacion):,} | variables: {X_ent.shape[1]}")
print(f"Épocas entrenadas: {len(perdida)} | mejor época: {mejor_epoca} | parámetros: {red.count_params():,}")
comparacion_nn

In [ ]:
c = comparacion_nn.set_index("modelo")
gana = "XGBoost" if c.loc["XGBoost", "precision_10"] >= c.loc["Red neuronal (Keras)", "precision_10"] else "la red neuronal"
display(Markdown(f"""
**Resultado** (validación por fecha desde {CORTE.date()}, mismas variables):
- Red neuronal: precisión en el 10% = **{c.loc['Red neuronal (Keras)','precision_10']:.3f}**, AUC = {c.loc['Red neuronal (Keras)','auc']:.3f}.
- XGBoost: precisión en el 10% = **{c.loc['XGBoost','precision_10']:.3f}**, AUC = {c.loc['XGBoost','auc']:.3f}.
- Gana **{gana}** por {abs(c.loc['XGBoost','precision_10'] - c.loc['Red neuronal (Keras)','precision_10'])*100:.1f} puntos en la métrica de negocio.
- La curva muestra que la pérdida de validación deja de mejorar en la época {mejor_epoca}; el early stopping detuvo el entrenamiento {len(perdida) - mejor_epoca} épocas después y restauró los pesos de la mejor época, evitando el sobreajuste.
- La red se entrenó con {len(ent_nn):,} pedidos y {X_ent.shape[1]} variables (después del one-hot).
""" + ("""
**Decisión:** mantenemos XGBoost en producción: gana en la métrica que le importa al negocio y además es más rápido, más fácil de explicar y más liviano para Render."""
       if gana == "XGBoost" else """
**Decisión:** aunque la red gana por poco en esta validación, la diferencia es pequeña y la red no es determinista en GPU (puede variar entre corridas). Mantenemos XGBoost en producción por ser estable, explicable y liviano para Render.""")))

**¿Por qué gana uno u otro en datos de tabla?** En datos tabulares de tamaño medio como este (pocas decenas de variables, muchas con relaciones de "umbral", como *llegó tarde sí/no* o *más de un vendedor*), los árboles con boosting suelen ganar o empatar con una red densa: cortan directamente por umbrales, manejan escalas y valores extremos sin transformarlos y necesitan poco ajuste. La red tiene que aprender esos cortes con funciones suaves, es sensible al escalado y a las variables one-hot dispersas, y con unas decenas de miles de filas (ver la salida) no tiene datos suficientes para aprovechar su flexibilidad, que brilla con datos no estructurados (texto, imágenes) como en la Parte 4. Por eso lo esperable en datos de tabla es un empate o una ventaja pequeña del boosting.

## Parte 5 · Despliegue

Versiones de las librerías de este Colab: el `requirements.txt` de la API se escribe **automáticamente** con estas mismas versiones, para que el `.joblib` cargue en Render.

In [ ]:
import sys, platform, sklearn, xgboost
print(f"pandas=={pd.__version__}\nnumpy=={np.__version__}\nscikit-learn=={sklearn.__version__}\nxgboost=={xgboost.__version__}")
print("Python", platform.python_version())

In [ ]:
# Librerías para escribir y probar la API dentro de Colab (TestClient necesita httpx)
!pip install -q fastapi uvicorn httpx
!pip show fastapi pydantic uvicorn httpx | grep -E "^(Name|Version)"
# Si este pip instaló o actualizó pydantic, reiniciar el entorno una vez y volver a ejecutar todo

In [ ]:
import os, shutil, joblib as _joblib, fastapi, pydantic, uvicorn
os.makedirs("api", exist_ok=True)

# xgboost-cpu es el mismo xgboost sin las librerías de GPU (instalación mucho más liviana en Render)
requisitos = f"""pandas=={pd.__version__}
numpy=={np.__version__}
scikit-learn=={sklearn.__version__}
xgboost-cpu=={xgboost.__version__}
joblib=={_joblib.__version__}
fastapi=={fastapi.__version__}
pydantic=={pydantic.__version__}
uvicorn=={uvicorn.__version__}
"""
with open("api/requirements.txt", "w") as f:
    f.write(requisitos)
with open("api/.python-version", "w") as f:
    f.write(platform.python_version() + "\n")
for archivo in ["preparacion.py", "modelo_riesgo.joblib", "umbral.json"]:
    shutil.copy(archivo, "api/")
print(requisitos)
print("Python para Render:", platform.python_version())

In [ ]:
%%writefile api/main.py
"""
API de riesgo de mala reseña — Proyecto final Olist.

GET  /        -> confirma que la API está viva
POST /riesgo  -> recibe las 17 columnas del contrato y devuelve la probabilidad de 1-2 estrellas
GET  /docs    -> documentación automática (Swagger) que genera FastAPI
"""
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator

CARPETA = Path(__file__).resolve().parent

# preparacion.py debe estar en esta misma carpeta: el pipeline lo necesita al cargarse.
modelo = joblib.load(CARPETA / "modelo_riesgo.joblib")
with open(CARPETA / "umbral.json", encoding="utf-8") as f:
    UMBRAL = float(json.load(f)["umbral"])

app = FastAPI(
    title="API Olist · Riesgo de mala reseña",
    description=(
        "Predice, en el momento de la entrega, la probabilidad de que un pedido reciba una "
        "reseña de 1 o 2 estrellas. `riesgo_alto` es true si la probabilidad alcanza el umbral "
        "del 10% de pedidos más riesgosos (calculado con los datos de entrenamiento)."
    ),
    version="1.0.0",
)

# El modelo predice en el momento de la entrega: compra, entrega real y fecha prometida son obligatorias.
FECHAS_OBLIGATORIAS = {"order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"}
CAMPOS_FECHA = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]


class Pedido(BaseModel):
    """Las 17 columnas de entrada del contrato (todas menos mala_resena)."""

    order_id: str = Field(..., min_length=1, examples=["d6cb7abec3164d194483906f3702748a"])
    order_purchase_timestamp: str = Field(..., examples=["2018-03-07 22:02:23"])
    order_approved_at: Optional[str] = Field(None, examples=["2018-03-09 02:35:30"])
    order_delivered_carrier_date: Optional[str] = Field(None, examples=["2018-03-09 20:03:25"])
    order_delivered_customer_date: str = Field(..., examples=["2018-04-11 20:07:37"])
    order_estimated_delivery_date: str = Field(..., examples=["2018-04-04 00:00:00"])
    estado_cliente: str = Field(..., min_length=2, max_length=2, examples=["CE"])
    n_items: int = Field(..., ge=1, examples=[1])
    n_vendedores: int = Field(..., ge=1, examples=[1])
    precio_total: float = Field(..., ge=0, examples=[420.0])
    flete_total: float = Field(..., ge=0, examples=[35.5])
    peso_total_g: Optional[float] = Field(None, ge=0, examples=[1200.0])
    fotos_promedio: Optional[float] = Field(None, ge=0, examples=[2.0])
    categoria: Optional[str] = Field(None, examples=["informatica_acessorios"])
    estado_vendedor: Optional[str] = Field(None, examples=["SP"])
    tipo_pago: Optional[str] = Field(None, examples=["credit_card"])
    cuotas_max: Optional[float] = Field(None, ge=0, examples=[3])

    @field_validator(*CAMPOS_FECHA)
    @classmethod
    def fecha_valida(cls, valor, info):
        if valor is None or str(valor).strip() == "":
            if info.field_name in FECHAS_OBLIGATORIAS:
                raise ValueError("esta fecha es obligatoria")
            return None
        try:
            pd.Timestamp(valor)
        except Exception:
            raise ValueError("fecha inválida; use el formato 'AAAA-MM-DD HH:MM:SS'")
        return valor

    @field_validator("estado_cliente", "estado_vendedor")
    @classmethod
    def sigla_mayuscula(cls, valor):
        return valor.upper() if isinstance(valor, str) else valor


@app.get("/")
def inicio():
    return {
        "estado": "ok",
        "mensaje": "API de riesgo de mala reseña de Olist activa.",
        "umbral_riesgo_alto": round(UMBRAL, 4),
        "documentacion": "/docs",
        "hora_servidor": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


@app.post("/riesgo")
def riesgo(pedido: Pedido):
    fila = pd.DataFrame([pedido.model_dump()])
    try:
        probabilidad = float(modelo.predict_proba(fila)[:, 1][0])
    except Exception as error:  # el modelo no pudo procesar la fila
        raise HTTPException(status_code=500, detail=f"Error al predecir: {error}")
    riesgo_alto = probabilidad >= UMBRAL
    return {
        "order_id": pedido.order_id,
        "probabilidad": round(probabilidad, 4),
        "riesgo_alto": bool(riesgo_alto),
        "umbral": round(UMBRAL, 4),
        "accion_sugerida": "Contactar al cliente" if riesgo_alto else "Sin acción",
    }

In [ ]:
# Prueba local de la API (antes de desplegar), con el cliente de pruebas de FastAPI
import importlib, sys
sys.path.insert(0, os.path.abspath("api"))
from fastapi.testclient import TestClient
import main as api_main
importlib.reload(api_main)
cliente = TestClient(api_main.app)

ejemplo = validacion[ENTRADAS].iloc[0].to_dict()
ejemplo = {k: (str(v) if isinstance(v, pd.Timestamp) else (None if pd.isna(v) else v)) for k, v in ejemplo.items()}

print("GET /            ->", cliente.get("/").status_code, cliente.get("/").json()["estado"])
r = cliente.post("/riesgo", json=ejemplo)
print("POST /riesgo     ->", r.status_code, r.json())
print("precio negativo  ->", cliente.post("/riesgo", json={**ejemplo, "precio_total": -5}).status_code)
sin_campo = {k: v for k, v in ejemplo.items() if k != "estado_cliente"}
print("falta un campo   ->", cliente.post("/riesgo", json=sin_campo).status_code)
print("fecha inválida   ->", cliente.post("/riesgo", json={**ejemplo, "order_purchase_timestamp": "ayer"}).status_code)
print("GET /docs        ->", cliente.get("/docs").status_code)
assert abs(r.json()["probabilidad"] - modelo_final.predict_proba(validacion[ENTRADAS].iloc[[0]])[0, 1]) < 1e-3

In [ ]:
# Datos para la app de Streamlit (carpeta app/datos)
os.makedirs("app/datos", exist_ok=True)

# Pedidos de ejemplo: del periodo de validación, con distintos niveles de riesgo
val_ej = validacion.copy()
val_ej["proba"] = modelo_final.predict_proba(val_ej[ENTRADAS])[:, 1]
val_ej["dias_retraso"] = dias_de_retraso(val_ej).round(0)
ejemplos = pd.concat([
    val_ej.nlargest(4, "proba"),
    val_ej[val_ej["proba"].between(umbral * 0.8, umbral * 1.2)].sample(3, random_state=1),
    val_ej.nsmallest(3, "proba"),
])
ejemplos = ejemplos.sample(frac=1, random_state=3)
ejemplos["etiqueta"] = ejemplos.apply(
    lambda r: f"{r.order_id[:8]}… · " + (f"retraso +{int(r.dias_retraso)} días" if r.dias_retraso > 0 else "a tiempo")
              + f" · {r.n_items} artículo(s) · {r.estado_cliente}", axis=1)
ejemplos[["etiqueta"] + ENTRADAS + ["mala_resena"]].to_csv("app/datos/ejemplos_pedidos.csv", index=False)

for archivo in ["sentimiento_muestra.csv", "desacuerdos_clasificados.csv", "temas_comentarios.csv",
                "segmentos_vendedores.csv", "perfil_segmentos.csv"]:
    shutil.copy(archivo, "app/datos/")
print(sorted(os.listdir("app/datos")))

Prueba de la API desplegada (debe devolver una probabilidad):

In [ ]:
import requests

URL_API = "https://SU-API.onrender.com"     # <- pegar aquí la URL de Render (la misma de api_url.txt)

pedido = pedidos[CONTRATO[:-1]].head(1).astype(object).where(pedidos[CONTRATO[:-1]].head(1).notna(), None).iloc[0].to_dict()
pedido = {k: (str(v) if isinstance(v, pd.Timestamp) else v) for k, v in pedido.items()}
if "SU-API" in URL_API:
    print("Falta pegar la URL de la API desplegada en URL_API.")
else:
    try:
        print("GET / ->", requests.get(URL_API + "/", timeout=90).json())
        print("POST /riesgo ->", requests.post(URL_API + "/riesgo", json=pedido, timeout=90).json())
    except Exception as error:
        print("La API no respondió (Render puede tardar ~1 minuto en despertar). Detalle:", error)

## Parte 6 · Ficha del modelo

La ficha va en `ficha_modelo.md` (también se muestra en la app). Aquí, solo el enlace o una copia.

La celda siguiente calcula los errores del modelo al llamar al 10% más riesgoso de la validación, y escribe `ficha_modelo.md` con las cifras de este notebook (así la ficha y las salidas siempre coinciden).

In [ ]:
from sklearn.metrics import confusion_matrix

proba_mejor = probas_val[mejor]
corte_val = np.quantile(proba_mejor, 0.90)
llamar = (proba_mejor >= corte_val).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val, llamar).ravel()
print(f"Validación ({len(y_val)} pedidos), llamando al 10% más riesgoso:")
print(f"  Llamadas útiles (verdaderos positivos): {tp}")
print(f"  Llamadas a clientes que no iban a quejarse (falsos positivos): {fp}")
print(f"  Malas reseñas que no se llamaron (falsos negativos): {fn}")
print(f"  Clientes satisfechos no llamados (verdaderos negativos): {tn}")
print(f"  Malas reseñas detectadas (recall): {tp/(tp+fn):.3f}")

# Riesgo promedio que el modelo asigna según el estado del vendedor: ¿castiga a regiones?
val_r = validacion.assign(proba=proba_mejor, retrasado=(dias_de_retraso(validacion) > 0).astype(int))
por_retraso = val_r.groupby("retrasado")["proba"].mean()
print(f"\nProbabilidad media: pedidos a tiempo {por_retraso.loc[0]:.3f} | pedidos retrasados {por_retraso.loc[1]:.3f}")

In [ ]:
# Desglose de exclusiones. Se usa un LEFT JOIN contra las reseñas agrupadas (rápido) en vez de un
# NOT EXISTS por fila, que en la base del curso supera el tiempo máximo por consulta (statement timeout).
desglose = pd.read_sql("""
SELECT o.order_status = 'delivered'                   AS entregado,
       o.order_delivered_customer_date IS NOT NULL    AS con_fecha_entrega,
       r.order_id IS NOT NULL                         AS con_resena,
       COUNT(*)                                       AS pedidos
FROM final.orders o
LEFT JOIN (SELECT DISTINCT order_id FROM final.order_reviews) r ON r.order_id = o.order_id
GROUP BY 1, 2, 3
""", motor)
excl = pd.Series({
    "total": desglose["pedidos"].sum(),
    "no_entregados": desglose.loc[~desglose["entregado"], "pedidos"].sum(),
    "sin_fecha_entrega": desglose.loc[desglose["entregado"] & ~desglose["con_fecha_entrega"], "pedidos"].sum(),
    "sin_resena": desglose.loc[desglose["entregado"] & desglose["con_fecha_entrega"] & ~desglose["con_resena"], "pedidos"].sum(),
}).astype(int)
n_total = int(excl["total"])
print(excl.to_string(), "\nQuedan:", n_total - excl["no_entregados"] - excl["sin_fecha_entrega"] - excl["sin_resena"], "(contrato:", len(pedidos), ")")
ficha = f"""# Ficha del modelo · Riesgo de mala reseña (Olist)

**Equipo:** Ruiz Melissa, Rodrigo y Cesar · **Tipo de modelo:** {mejor} (un conjunto de árboles de decisión que aprende de pedidos pasados). Archivos técnicos: `modelo_riesgo.joblib` y `preparacion.py`.

## 1. Para qué sirve (y para qué no)
- **Sirve para:** ordenar los pedidos **recién entregados** según la probabilidad de que el cliente deje una reseña de 1 o 2 estrellas, para que atención al cliente llame primero al **10% más riesgoso** (su capacidad diaria) y resuelva el problema antes de la reseña.
- **No debe usarse para:** castigar, sancionar o calificar vendedores o transportistas; decidir reembolsos o beneficios de forma automática; predecir antes de la entrega (necesita la fecha real de entrega); ni para pedidos cancelados o no entregados.

## 2. Datos
- **Origen:** base de datos pública de Olist (esquema `final` del curso), pedidos reales de Brasil.
- **Periodo:** compras del {pedidos['order_purchase_timestamp'].min().date()} al {pedidos['order_purchase_timestamp'].max().date()}.
- **Cuántos:** {len(pedidos):,} pedidos entregados y con reseña, de {n_total:,} pedidos en la tabla de pedidos. Tasa de mala reseña: {pedidos['mala_resena'].mean()*100:.1f}%.
- **Qué se excluyó y por qué:** {excl['no_entregados']:,} pedidos no entregados (cancelados, en camino, etc.) y {excl['sin_fecha_entrega']:,} sin fecha de entrega registrada, porque el modelo predice *en el momento de la entrega*; y {excl['sin_resena']:,} pedidos entregados sin reseña, porque no sabemos cómo terminaron. Tampoco se usa nada de la reseña (texto, fechas, puntaje) como variable: esa información no existe cuando el modelo hace la predicción.

## 3. Qué tan bien funciona
Validación honesta por fecha: entrenamos con pedidos anteriores al {CORTE.date()} y probamos con {len(validacion):,} pedidos posteriores que el modelo no había visto.
- **De cada 10 clientes que llamamos, unos {p_mod*10:.1f} iban a dejar una mala reseña** (tasa de acierto al llamar al 10% más riesgoso: {p_mod:.3f}).
- Si se llamara **al azar**, solo unos {p_azar*10:.1f} de cada 10 ({p_azar:.3f}).
- Si se llamara **a los más retrasados** (regla simple), unos {p_regla*10:.1f} de cada 10 ({p_regla:.3f}).
- En un periodo con **pocos retrasos** (abril–mayo 2018) el modelo bajó a {p2_mod*10:.1f} de cada 10 ({p2_mod:.3f}), mientras que la regla bajó a {p2_regla*10:.1f} ({p2_regla:.3f}).

## 4. Límites conocidos
- Depende mucho del **retraso**: si la logística mejora o cambia la forma de calcular la fecha prometida, la precisión baja (ya lo vimos en abril–mayo).
- No ve el **producto** en sí (calidad, si coincide con el anuncio) ni el trato del vendedor: muchas malas reseñas por "producto distinto" o "defectuoso" no son detectables con estos datos.
- Se entrenó con 2016–mayo 2018; patrones nuevos (huelgas, nuevas regiones o categorías, Black Friday) pueden sorprenderlo.
- Es una **probabilidad**, no una certeza: en validación detecta {tp/(tp+fn)*100:.0f}% de las malas reseñas al llamar solo al 10%.

## 5. A quién afecta un error
- **Falso positivo** (llamamos a un cliente que estaba satisfecho, {fp:,} casos en validación): costo de una llamada y una pequeña molestia para el cliente. Es un error barato.
- **Falso negativo** (no llamamos a un cliente que sí deja 1–2 estrellas, {fn:,} casos en validación): se pierde la oportunidad de resolver el problema; afecta la reputación del vendedor y de Olist.
- **Riesgo de mal uso:** como el retraso pesa tanto (probabilidad media de {por_retraso.loc[1]:.2f} en pedidos retrasados vs {por_retraso.loc[0]:.2f} a tiempo), usar el puntaje para **sancionar vendedores** sería injusto: muchos retrasos son del transportista o de la región del cliente, no del vendedor. Tampoco debe usarse para dar un trato peor a clientes de regiones lejanas.

## 6. Recomendación
- Usarlo como **lista de prioridad diaria**: ordenar los pedidos entregados del día por probabilidad y llamar al 10% de arriba (lo que alcanza la capacidad del equipo). La app marca "riesgo alto" cuando la probabilidad supera {umbral*100:.0f}%, el corte del 10% más riesgoso en los datos históricos; como la cantidad de retrasos cambia de un mes a otro, ese corte puede marcar más o menos del 10% en un día dado, así que manda el orden, no el corte. La persona de atención decide qué ofrecer; el modelo no toma decisiones sobre el cliente.
- **Supervisión humana:** revisar cada semana una muestra de llamadas para ver si el problema era real, y registrar el resultado para reentrenar.
- **Revisión:** medir la precisión en el 10% **cada mes** con las reseñas que llegaron; reentrenar y recalcular el umbral cada trimestre, o antes si la tasa de retraso cambia mucho o la precisión cae por debajo de la regla del retraso.
"""
with open("ficha_modelo.md", "w", encoding="utf-8") as f:
    f.write(ficha)
shutil.copy("ficha_modelo.md", "app/ficha_modelo.md")
display(Markdown(ficha))

## Uso de IA

Indiquen qué herramientas de IA usaron y para qué. Todo lo que entregan lo deben poder explicar en la defensa.

| Herramienta | Para qué la usamos | Qué revisamos nosotros |
|---|---|---|
| Gemini en Colab | Sugerencias de código para gráficos y depuración de errores puntuales en nuestra primera versión. | Que el código corriera y los gráficos mostraran lo que decimos. |
| Claude (Anthropic) | Auditoría de nuestra primera versión contra el enunciado y la rúbrica; reorganización del notebook; código de la API (FastAPI + Pydantic), de la app de Streamlit y de la ficha; respuestas generadas desde las salidas; guía de despliegue en GitHub, Render y Streamlit. | Revisamos y ejecutamos todo el código. La consulta SQL, la primera versión del modelo, la segmentación y el análisis de sentimiento partieron de nuestro trabajo; las decisiones (variables, corte de validación, k, clasificación de desacuerdos) las discutimos y las podemos explicar. Las cifras salen siempre de las salidas del notebook. |

Ninguna herramienta de IA tuvo acceso a la contraseña de la base de datos dentro del notebook: la celda de conexión la pide con `getpass`.

In [ ]:
# Empaquetar los archivos para la entrega (el .ipynb se descarga aparte: Archivo › Descargar › .ipynb)
import zipfile
archivos = ["preparacion.py", "modelo_riesgo.joblib", "umbral.json", "ficha_modelo.md", "sentimiento_muestra.csv",
            "desacuerdos_clasificados.csv", "segmentos_vendedores.csv", "perfil_segmentos.csv"]
with zipfile.ZipFile("artefactos_proyecto.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a)
    for carpeta in ["api", "app"]:
        for raiz, _, nombres_arch in os.walk(carpeta):
            for n in nombres_arch:
                if "__pycache__" not in raiz:
                    z.write(os.path.join(raiz, n))
print("Listo: artefactos_proyecto.zip con", len(zipfile.ZipFile("artefactos_proyecto.zip").namelist()), "archivos")